# Mini Project 1 — Analysis Notebook

**Your name:** Gena Lee

**Dataset:**  Last.fm_data.json

**Date:**  5/6/2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [20]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


---

## Section 1 — Overview

Before writing any code, fill in this section. A good Overview tells anyone reading your notebook — including a future employer — what the analysis is about before they see a single chart.

**Dataset:** My data comes from the Last.fm API (last.fm/api), a free public API that aggregates real listening behavior from millions of users across streaming platforms like Spotify and Apple Music. Last.fm tracks individual song plays and exposes aggregated statistics through its API.

I will access the data using Python's requests library, pulling JSON responses directly into pandas DataFrames. No authentication beyond a free API key is required for the public read-only endpoints I plan to use.

**Why this dataset:** *I chose the Last.fm API because of my interests in music and how it relates on a global scale. I think it would be super interesting to see how music is perceived and listened to across countries. I would also be open to focusing on specific countries (e.g. US, Korea, Brazil) if I have time.*

**Three analytical questions:**

1. *Which artist has the most devoted fanbase?*
2. *Which genres have the most listeners?*
3. *Among the top 100 global artists, which genres have the highest play-to-listener ratio?*


**What a practitioner would do with these findings:** *(One sentence. Who uses this, and for what?)*

---

## Section 2 — Data Profile

Load your dataset and get a basic picture of what's in it. Answer these questions in a markdown cell below your code:

- How many rows and columns does your dataset have?
- What does each column represent?
- Are there any obvious data quality issues (missing values, unexpected types, inconsistent formatting)?
- Which column or columns will your analysis focus on, and why?

In [22]:
import json
from pathlib import Path

with open("Last.fm_Data.json", "r", encoding="utf-8") as f:
    root = json.load(f)

print("Top-level keys:")
for k, v in root.items():
    if isinstance(v, dict) and "records" in v:
        print(f"  {k:25s}  ({len(v['records'])} rows, columns={v['columns']})")
    else:
        print(f"  {k:25s}  ({type(v).__name__})")

Top-level keys:
  _snapshot_meta             (dict)
  df                         (60 rows, columns=['country_label', 'country_param', 'name', 'listeners', 'url', 'streamable', 'image', '@attr.rank', 'mbid'])
  names_by_country           (10 rows, columns=['rank', 'Brazil', 'South Korea', 'United States'])
  pairwise_df                (3 rows, columns=['profile_country_a', 'profile_country_b', 'overlap_artists', 'jaccard_similarity'])
  unique_df                  (3 rows, columns=['profile_country', 'artists_only_on_this_chart', 'examples'])
  rq1_overlap_summary        (1 rows, columns=['key', 'count', 'names_csv'])
  tags_df                    (50 rows, columns=['name', 'url', 'reach', 'taggings', 'streamable'])
  chart100_df                (99 rows, columns=['name', 'playcount', 'listeners', 'mbid', 'url', 'streamable', 'image'])
  rq3_artists_df             (98 rows, columns=['name', 'mbid', 'playcount', 'listeners', 'play_per_listener', 'genre_tag'])
  rq3_genre_summary          (5

In [26]:
import pandas as pd

#root is the json file, which I labeled in the previous cell
#I am relabeling the df section as block, which has a list of columns and records
block = root["df"]
#I am creating a dataframe from the records and columns in the block
df = pd.DataFrame(block["records"], columns=block["columns"])
#I am printing the shape and head of the dataframe
print(df.shape)
df.head(100)

(60, 9)


,country_label,country_param,name,listeners,url,streamable,image,@attr.rank,mbid
0,United States,United States,Kanye West,242926,https://www.last.fm/music/Kanye+West,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,1,NaN
1,United States,United States,Drake,238213,https://www.last.fm/music/Drake,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,2,9fff2f8a-21e6-47de-a2b8-7f449929d43f
2,United States,United States,Kendrick Lamar,228488,https://www.last.fm/music/Kendrick+Lamar,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,3,381086ea-f511-4aba-bdf9-71c753dc5077
3,United States,United States,"Tyler, The Creator",228055,"https://www.last.fm/music/Tyler,+The+Creator",0,[{'#text': 'https://lastfm.freetls.fastly.net/...,4,f6beac20-5dfe-4d1f-ae02-0b0a740aafd6
4,United States,United States,PinkPantheress,214472,https://www.last.fm/music/PinkPantheress,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,5,7441014f-f8f5-494f-81db-ff166fbc078d
5,United States,United States,Radiohead,206407,https://www.last.fm/music/Radiohead,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,6,a74b1b7f-71a5-4011-9441-d0b5e4122711
6,United States,United States,The Weeknd,198587,https://www.last.fm/music/The+Weeknd,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,7,c8b03190-306c-4120-bb0b-6f2ebfc06ea9
7,United States,United States,Rihanna,198126,https://www.last.fm/music/Rihanna,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,8,73e5e69d-3554-40d8-8516-00cb38737a1c
8,United States,United States,Tame Impala,191412,https://www.last.fm/music/Tame+Impala,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,9,63aa26c3-d59b-4da4-84ac-716b54f1ef4d
9,United States,United States,Michael Jackson,186669,https://www.last.fm/music/Michael+Jackson,0,[{'#text': 'https://lastfm.freetls.fastly.net/...,10,f27ec8db-af05-4f36-916e-3d57f91ecf5e


In [16]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual filename.
# The file should be in the same folder as this notebook.
# If you're loading from an API result, replace pd.read_csv() with the appropriate call.
#
# Example (app review dataset from class):
# df = pd.read_csv('app_reviews_demo.csv')

df = pd.read_json('Last.fm_data.json')  # ← replace with your filename

# Rank by playcount
df = df.sort_values("playcount", ascending=False).reset_index(drop=True)
df["rank_by_playcount"] = df.index + 1

print(df.shape)
df.head()

KeyError: 'playcount'

In [13]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype
---  ------             --------------  -----
 0   rank               50 non-null     int64
 1   name               50 non-null     str  
 2   playcount          50 non-null     int64
 3   listeners          50 non-null     int64
 4   mbid               49 non-null     str  
 5   url                50 non-null     str  
 6   rank_by_playcount  50 non-null     int64
dtypes: int64(4), str(3)
memory usage: 2.9 KB


In [14]:
# Summary statistics for numeric columns
df.describe()

,rank,playcount,listeners,rank_by_playcount
count,50.00000,5.000000e+01,5.000000e+01,50.00000
mean,25.50000,7.552190e+08,4.692388e+06,25.50000
std,14.57738,7.126519e+08,1.895089e+06,14.57738
min,1.00000,8.923720e+07,1.334852e+06,1.00000
25%,13.25000,3.641394e+08,3.269678e+06,13.25000
50%,25.50000,5.831828e+08,4.308792e+06,25.50000
75%,37.75000,8.265084e+08,5.877836e+06,37.75000
max,50.00000,3.838525e+09,9.121759e+06,50.00000


**Your data profile notes:**  
*I found that the original ranking of the top artists weren't ranked by playcount or listeners. So, I had to write a new code to rank the artists by playcount, which gave me a more accurate ranking.*

---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *(paste your first research question from MP1a here)*

In [28]:
# Your analysis for Question 1


**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *(paste your second research question here)*

In [29]:
# Your analysis for Question 2


**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *(paste your third research question here)*

In [30]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [31]:
# Your visualization
# Example structure — replace with your actual columns and finding

# fig = px.bar(
#     df,
#     x="your_category_column",
#     y="your_value_column",
#     title="Your finding stated as a claim",
#     labels={"your_category_column": "Readable label", "your_value_column": "Readable label"}
# )
# fig.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.